# GGUF Q5_K_M + Q6_K — export & benchmark (standalone, parallel account)

**Run this on a SECOND Kaggle account, in parallel with the chat run on account 1.** It does ONLY
the GGUF quants: export Q5_K_M + Q6_K on Kaggle (no Colab), then run the same authentic chats + a
completion pass through each quant so you can **eyeball reply quality and see tok/s (the speed win)**.

**Order:** Cell 1 (config) → Cell 2 (clone + deps + Stockfish) → Cell 3 (download base + adapter,
~9GB) → **GGUF EXPORT** (~6-10 min) → **GGUF EYEBALL** (chats + completion per quant) → chart.

Final quants land in `/kaggle/working/gguf` (downloadable to serve on the 4060). The chart here is
QUANT-ONLY (q5/q6 + the nf4/base seeds); to get the FULL cross-model chart, copy this account's
`docs/findings/report_assets/measured-e4b-q5.json` + `-q6.json` next to account 1's `measured-*.json`
and rebuild. Needs an `HF_TOKEN` Kaggle secret.


In [ ]:
# Cell 1 - config (GGUF quant export + benchmark; standalone)
import os
BASE_REPO    = 'unsloth/gemma-4-E4B-it'              # the base the adapter was trained on
REPO_URL     = 'https://github.com/RyanDev1st/llm-and-engine.git'
BRANCH       = 'feat/gguf-quant-bench'               # THIS branch (has the export pipeline)
ADAPTER_REPO = 'RyanDev1st/gemma4-chesscoach-ckpt-v4'
TAG          = 'best'
WORKDIR      = '/kaggle/working/llm-and-engine'
BASE_DIR     = '/kaggle/working/gemma4_e4b'
ADAPTER_DIR  = '/kaggle/working/adapter/best'
ASSETS_DIR   = f'{WORKDIR}/docs/findings/report_assets'
os.environ['CHESS_SF_TIMEOUT'] = '2.0'               # cap per-analysis engine wait for the chats
# --- GGUF export: merged + f16 -> /kaggle/temp (big scratch); final quants -> /kaggle/working ------
RUN_GGUF_EXPORT = True
GGUF_QUANTS  = ['Q5_K_M', 'Q6_K']                    # both reuse ONE f16 merge
GGUF_OUT_DIR = '/kaggle/working/gguf'
GGUF_SCRATCH = '/kaggle/temp/ggufx'
# --- benchmark the quants: authentic chats (eyeball + tok/s) + OOD completion (quality numbers) ----
RUN_GGUF_AB     = True
BUILD_MODEL_LINES = True
print(f'GGUF quant bench | quants {GGUF_QUANTS} | export={RUN_GGUF_EXPORT} | bench={RUN_GGUF_AB}')


In [ ]:
# Cell 2 - clone repo (prints HEAD so the commit is visible) + deps + Stockfish
import subprocess, sys, shutil
env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
if not os.path.exists(WORKDIR):
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,WORKDIR], check=True, env=env)
else:
    subprocess.run(['git','-C',WORKDIR,'fetch','origin',BRANCH], check=True, env=env)
    subprocess.run(['git','-C',WORKDIR,'reset','--hard',f'origin/{BRANCH}'], check=True, env=env)
print('HEAD:', subprocess.run(['git','-C',WORKDIR,'log','-1','--oneline'], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers','accelerate','peft','bitsandbytes','sentencepiece','protobuf','python-chess'], check=True)
os.environ['PYTHONPATH'] = f'{WORKDIR}/src/llm'
# Stockfish — needed by the CHESS completion eval (Cell 6.7b). backend.engine._resolve_sf() finds it
# on PATH or /usr/games/stockfish; we also export CHESS_SF so the subprocess resolves it deterministically.
sf = shutil.which('stockfish')
if not sf:
    subprocess.run(['bash','-lc','apt-get update -qq && apt-get install -y -qq stockfish'], check=False)
    sf = shutil.which('stockfish') or ('/usr/games/stockfish' if os.path.exists('/usr/games/stockfish') else None)
if sf:
    os.environ['CHESS_SF'] = sf
    print('stockfish:', sf)
else:
    print('stockfish: NOT FOUND -> chess completion (Cell 6.7b) will skip; OOD stress still runs')
import transformers, peft, bitsandbytes
print('transformers', transformers.__version__, '| peft', peft.__version__, '| bnb', bitsandbytes.__version__)

In [ ]:
# Cell 3 - download the E4B base (~9GB) + the v4 adapter (the merge source)
from huggingface_hub import login, snapshot_download
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret('HF_TOKEN'))
except Exception:
    login(os.environ['HF_TOKEN'])
snapshot_download(repo_id=BASE_REPO, local_dir=BASE_DIR,
                  allow_patterns=['*.json','*.safetensors','*.model','*.txt','tokenizer*'])
snapshot_download(repo_id=ADAPTER_REPO, local_dir='/kaggle/working/adapter', allow_patterns=[f'{TAG}/*'])
need = ('adapter_model.safetensors', 'adapter_config.json')
assert all(os.path.exists(f'{ADAPTER_DIR}/{f}') for f in need), f'adapter not at {ADAPTER_DIR}'
print('e4b base + adapter ready:', os.listdir(ADAPTER_DIR))


In [ ]:
# === GGUF EXPORT (Kaggle) - merge -> f16 -> Q5_K_M + Q6_K (no Colab) =============================
# Ported from the proven Colab cell (PR #26): prebuilt llama-quantize (no slow build), converter from
# a shallow llama.cpp clone, torchao-conflict fix (uninstall + merge in a FRESH subprocess). Kaggle
# adaptation: intermediates -> GGUF_SCRATCH (/kaggle/temp), final quants -> GGUF_OUT_DIR
# (/kaggle/working). Each step prints; never a silent hang. First run ~6-10 min (the bf16 merge
# dominates); the 2nd quant reuses the f16 (~1-2 min). Needs Cell 3 (base + adapter) done.
import os, sys, time, glob, json as _json, shutil, zipfile, urllib.request, subprocess
from pathlib import Path
def sh(c, **kw): print('  $', c, flush=True); subprocess.run(c, shell=True, check=True, **kw)
if not RUN_GGUF_EXPORT:
    print('RUN_GGUF_EXPORT=False -> skipping the export. Point the EYEBALL cell at existing .gguf paths if you have them.')
    Q5_PATH = Q6_PATH = ''
else:
    t0 = time.time()
    os.makedirs(GGUF_SCRATCH, exist_ok=True); os.makedirs(GGUF_OUT_DIR, exist_ok=True)
    os.environ['CHESS_HF_BASE'] = BASE_DIR
    os.environ['CHESS_MERGE_DEVICE'] = 'auto'   # shard the ~16GB MULTIMODAL E4B across BOTH T4s (one 16GB card OOMs)
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # reduce CUDA fragmentation
    print('[1/5] converter deps (gguf/sentencepiece/protobuf)...', flush=True)
    sh('pip -q install gguf sentencepiece protobuf')
    LC = f'{GGUF_SCRATCH}/llamacpp'
    if not Path(LC + '/convert_hf_to_gguf.py').exists():
        print('[2/5] fetching llama.cpp converter + gguf-py...', flush=True)
        sh(f'git clone --depth 1 https://github.com/ggerganov/llama.cpp {LC}')
        sh(f'pip -q install {LC}/gguf-py')
    print('[3/5] prebuilt llama-quantize (download ubuntu-x64; build only as fallback)...', flush=True)
    QBIN = None
    try:
        rel = _json.loads(urllib.request.urlopen(
            'https://api.github.com/repos/ggerganov/llama.cpp/releases/latest', timeout=30).read())
        url = next(a['browser_download_url'] for a in rel['assets']
                   if 'ubuntu-x64' in a['name'] and a['name'].endswith('.zip'))
        sh(f"wget -q -O {GGUF_SCRATCH}/lq.zip '{url}'")
        with zipfile.ZipFile(f'{GGUF_SCRATCH}/lq.zip') as z: z.extractall(f'{GGUF_SCRATCH}/lq')
        hits = glob.glob(f'{GGUF_SCRATCH}/lq/**/llama-quantize', recursive=True)
        if hits:
            QBIN = hits[0]
            os.environ['LD_LIBRARY_PATH'] = str(Path(QBIN).parent) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
            subprocess.run(f"chmod +x '{QBIN}'", shell=True)
            if subprocess.run(f"'{QBIN}' --help", shell=True).returncode != 0:
                print('   prebuilt wont run (lib mismatch) -> build', flush=True); QBIN = None
            else: print('   prebuilt ready:', QBIN, flush=True)
    except Exception as e:
        print('   prebuilt fetch failed:', repr(e), flush=True)
    if not QBIN:
        QBIN = f'{LC}/build/bin/llama-quantize'
        if not Path(QBIN).exists():
            print('   building llama-quantize (GGML_CUDA=OFF)...', flush=True)
            sh(f'cmake -S {LC} -B {LC}/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_SERVER=OFF')
            sh(f'cmake --build {LC}/build --target llama-quantize -j $(nproc)')
    merged = f'{GGUF_SCRATCH}/merged'; f16 = f'{GGUF_SCRATCH}/f16.gguf'
    if not Path(f16).exists():
        print('[4/5] merge adapter -> bf16 (slow) + convert to f16...', flush=True)
        # peft requires torchao>=0.16 and RAISES on merge_and_unload if older; torchao is irrelevant to
        # the merge -> uninstall + merge in a FRESH subprocess (an in-kernel uninstall wont take once
        # peft imported torchao). export_gguf.merge loads bf16 low_cpu_mem_usage on CPU.
        subprocess.run('pip -q uninstall -y torchao', shell=True)
        mcode = ("import sys,os; sys.path.insert(0, r'%s/src/llm'); os.environ['CHESS_HF_BASE']=r'%s'; "
                 "from pathlib import Path; from llm_training.export_gguf import merge; "
                 "merge(Path(r'%s'), Path(r'%s'))" % (WORKDIR, BASE_DIR, ADAPTER_DIR, merged))
        subprocess.run([sys.executable, '-c', mcode], check=True)
        subprocess.run([sys.executable, f'{LC}/convert_hf_to_gguf.py', merged,
                        '--outfile', f16, '--outtype', 'f16'], check=True)
        shutil.rmtree(merged, ignore_errors=True)   # free the bf16 merge immediately
    Q5_PATH = Q6_PATH = ''
    for q in GGUF_QUANTS:
        out = f'{GGUF_OUT_DIR}/gemma4-E4B-chesscoach-{q}.gguf'
        if not Path(out).exists():
            print(f'[5/5] quantize -> {q} ...', flush=True)
            subprocess.run([QBIN, f16, out, q], check=True)
        sz = os.path.getsize(out) // (1 << 20)
        print(f'   {q}: {out}  ({sz} MB)', flush=True)
        if q == 'Q5_K_M': Q5_PATH = out
        if q == 'Q6_K': Q6_PATH = out
    Path(f16).unlink(missing_ok=True)   # free the f16 once both quants are made
    print(f'GGUF export done ({time.time()-t0:.0f}s). Q5={Q5_PATH} Q6={Q6_PATH}', flush=True)


In [ ]:
# === GGUF EYEBALL - quant quality + speed (run after the export) ================================
# Runs the SAME authentic chats through each LOCAL quant so you can read Q5_K_M vs Q6_K replies and
# see tok/s (the speed win), then a completion pass for the quality numbers. Tagged e4b-q5 / e4b-q6 so
# both land on the cross-model line chart next to nf4. Gated by RUN_GGUF_AB.
import os, glob, shutil, subprocess, sys
os.environ['CHESS_HF_BASE'] = BASE_DIR
env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
if not RUN_GGUF_AB:
    print('RUN_GGUF_AB=False -> skipping the GGUF eyeball (line chart shows the quants as n/a).')
else:
    # llama-cpp-python is REQUIRED to load a GGUF (GGUFModel imports llama_cpp). Wheel-only so it
    # NEVER source-compiles; try CUDA tags first (GPU speed), fall back to the CPU wheel (quality is
    # identical, but tok/s would then be CPU-speed -- NOT representative of the 4060).
    def _have_llama():
        try:
            import llama_cpp; return True
        except Exception:
            return False
    if not _have_llama():
        for idx in ('cu124', 'cu123', 'cu122', 'cu121'):
            subprocess.run('pip -q install --only-binary=:all: llama-cpp-python '
                           f'--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/{idx}', shell=True)
            if _have_llama(): print('llama-cpp-python: CUDA wheel', idx, flush=True); break
        if not _have_llama():
            subprocess.run('pip -q install --only-binary=:all: llama-cpp-python', shell=True)
            print('llama-cpp-python: CPU wheel (GGUF on CPU -> tok/s is CPU-speed, NOT the 4060)'
                  if _have_llama() else 'WARNING: NO llama-cpp-python wheel -> GGUF chats SKIP', flush=True)
    quants = [('e4b-q5', globals().get('Q5_PATH','') or f'{GGUF_OUT_DIR}/gemma4-E4B-chesscoach-Q5_K_M.gguf'),
              ('e4b-q6', globals().get('Q6_PATH','') or f'{GGUF_OUT_DIR}/gemma4-E4B-chesscoach-Q6_K.gguf')]
    from IPython.display import Markdown, display
    for tag, gguf in quants:
        if not gguf or not os.path.exists(gguf):
            print(f'{tag}: no file at {gguf} -> skip'); continue
        # eyeball the chats (real replies + tok/s); feeds tok/s to the chart
        sc = [sys.executable, '-m', 'llm_training.report.chat_showcase', '--gguf', gguf, '--tag', tag]
        print('==== chats:', ' '.join(sc), '====', flush=True)
        subprocess.run(sc, cwd=f'{WORKDIR}/src/llm', env=env, check=False)
        for f in sorted(glob.glob(f'{WORKDIR}/docs/findings/*-chat-showcase.md'))[-1:]:
            shutil.copy(f, f'/kaggle/working/{tag}-chat-showcase.md'); display(Markdown(open(f).read()))
        # quality numbers (completed/grounded) for the chart
        co = [sys.executable, '-m', 'llm_training.eval_completion', '--gguf', gguf, '--stress',
              '--time-budget', '1200', '--tag', tag]
        print('==== completion:', ' '.join(co), '====', flush=True)
        subprocess.run(co, cwd=f'{WORKDIR}/src/llm', env=env, check=False)


In [ ]:
# REPORT - CROSS-MODEL LINE CHART (run LAST). Collects every measured-*.json the cells above wrote
# (verb from the confusion cell, completion/grounded from Cell 6.7, tok/s from the chats, the GGUF
# quants from the A/B if you ran it) onto the model seeds and renders the line across ALL models:
# E2B prior -> E4B base -> E4B nf4 (current) -> E4B Q5_K_M -> E4B Q6_K. A model with no measured
# number is drawn as a GAP, never faked.
import os, subprocess, sys
env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
if not BUILD_MODEL_LINES:
    print('BUILD_MODEL_LINES=False -> skipping.')
else:
    pycode = ('from pathlib import Path;'
              'from llm_training.report import chart_data as D, ppt_charts, measured;'
              'm=measured.collect(r"%s");'
              'models=D.merge_measured(D.MODELS, m);'
              'ppt_charts.model_lines(models, Path(r"%s")/"chart-model-lines.png");'
              'print(D.model_table_md(models));'
              'print("measured tags:", sorted(m))') % (ASSETS_DIR, ASSETS_DIR)
    subprocess.run([sys.executable, '-c', pycode], cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    from IPython.display import Image, display
    p = f'{ASSETS_DIR}/chart-model-lines.png'
    if os.path.exists(p): print(p); display(Image(filename=p))
